In [1]:
!pip install -q -U transformers accelerate requests

import os
import re
import json
import glob
import torch
from collections import Counter
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0

In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = hidden_dim // n_head
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 32 heads, 128 head dim


In [3]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

--2026-10-01 16:59:47--  https://cs.stanford.edu/people/rak248/VG_100K/images.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9730308001 (9.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K.zip’

/kaggle/working/VG_ 100%[===================>]   9.06G  8.88MB/s    in 15m 52s 

2026-10-01 17:15:39 (9.75 MB/s) - ‘/kaggle/working/VG_100K.zip’ saved [9730308001/9730308001]

--2026-10-01 17:15:40--  https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5471658058 (5.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K_2.zip’

/kaggle/working/VG_ 100%[===================>]   5.10G  9.32MB/s    in 9m 36s  

2026-10-01 17:25:16 (9.06 MB

In [4]:
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    elif qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [5]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
print(len(all_questions), "questions loaded")

17492 questions loaded


In [6]:
def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


SOURCES = [
    ("attribute/amber", "attribute", "amber_"),
    ("object/pope", "object", "pope_"),
    ("relation/amber", "relation", "amber_"),
    ("relation/reefknot", "relation", "reefknot_"),
]
N_PER_SOURCE = 12

answer_ids = {"yes": [], "no": []}
samples = {}
for label, t, prefix in SOURCES:
    picked = []
    for path in sorted(glob.glob(f"{INPUT}/question-aware-cache-{t}/{t}_batch_*.pt"), key=batch_number):
        batch = torch.load(path, mmap=True)
        for e in batch:
            answer = e["answer_text"].strip().lower()
            if answer in answer_ids:
                answer_ids[answer].append(int(e["answer_logits"].argmax()))
            if e["question_id"].startswith(prefix) and len(picked) < N_PER_SOURCE:
                picked.append({
                    "question_id": e["question_id"],
                    "answer_text": e["answer_text"],
                    "answer_logits": e["answer_logits"].clone(),
                })
        del batch
        if len(picked) == N_PER_SOURCE:
            break
    samples[label] = picked

YES_ID = Counter(answer_ids["yes"]).most_common(1)[0][0]
NO_ID = Counter(answer_ids["no"]).most_common(1)[0][0]
print(f"yes_id={YES_ID} purity={answer_ids['yes'].count(YES_ID) / len(answer_ids['yes']):.4f} | "
      f"no_id={NO_ID} purity={answer_ids['no'].count(NO_ID) / len(answer_ids['no']):.4f}")

assert all(len(v) == N_PER_SOURCE for v in samples.values()), {k: len(v) for k, v in samples.items()}
for entries in samples.values():
    for e in entries:
        image_path = resolve_image_path(question_by_id[e["question_id"]])
        assert image_path is not None and os.path.exists(image_path), f"missing image for {e['question_id']}"
print({k: len(v) for k, v in samples.items()})

yes_id=3869 purity=1.0000 | no_id=1939 purity=1.0000
{'attribute/amber': 12, 'object/pope': 12, 'relation/amber': 12, 'relation/reefknot': 12}


In [7]:
u = ln_f.weight.float().cpu() * (lm_head.weight[YES_ID] - lm_head.weight[NO_ID]).float().cpu()
_u_cache = {}


def u_on(device):
    if device not in _u_cache:
        _u_cache[device] = u.to(device)
    return _u_cache[device]


with torch.no_grad():
    head_proj = []
    for layer in layers:
        w = layer.self_attn.o_proj.weight
        head_proj.append(w.float().T.contiguous().view(n_head, head_dim, hidden_dim) @ u_on(w.device))

_dla_cache = {}


def _embed_hook(module, args, kwargs):
    h = args[0] if args else kwargs["hidden_states"]
    _dla_cache["embed"] = h[0, -1].float() @ u_on(h.device)


def _make_attn_hook(l):
    def hook(module, args):
        z = args[0][0, -1].float().reshape(n_head, head_dim)
        _dla_cache[f"attn_{l}"] = (z * head_proj[l]).sum(-1)
    return hook


def _make_mlp_hook(l):
    def hook(module, args, output):
        _dla_cache[f"mlp_{l}"] = output[0, -1].float() @ u_on(output.device)
    return hook


def _final_norm_hook(module, args):
    h = args[0][0, -1].float()
    _dla_cache["resid_dot"] = h @ u_on(h.device)
    _dla_cache["rms"] = torch.sqrt(h.pow(2).mean() + ln_f.variance_epsilon)


_hook_handles = [layers[0].register_forward_pre_hook(_embed_hook, with_kwargs=True)]
for l, layer in enumerate(layers):
    _hook_handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_make_attn_hook(l)))
    _hook_handles.append(layer.mlp.register_forward_hook(_make_mlp_hook(l)))
_hook_handles.append(ln_f.register_forward_pre_hook(_final_norm_hook))
print(len(_hook_handles), "hooks registered")

66 hooks registered


In [8]:
@torch.no_grad()
def dla_question(image_path, question_text):
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)

    _dla_cache.clear()
    outputs = model(
        input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
        attention_mask=inputs["attention_mask"],
    )
    last_logits = outputs.logits[0, -1].float().cpu()

    return {
        "embed": _dla_cache["embed"].cpu(),
        "attn": torch.stack([_dla_cache[f"attn_{l}"].cpu() for l in range(n_layer)]),
        "mlp": torch.stack([_dla_cache[f"mlp_{l}"].cpu() for l in range(n_layer)]),
        "resid_dot": _dla_cache["resid_dot"].cpu(),
        "rms": _dla_cache["rms"].cpu(),
        "logit_diff": last_logits[YES_ID] - last_logits[NO_ID],
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
    }

In [9]:
results = {label: [] for label in samples}
rows = []

for label, entries in samples.items():
    for e in entries:
        q = question_by_id[e["question_id"]]
        r = dla_question(resolve_image_path(q), q["question_text"])
        total = r["embed"] + r["attn"].sum() + r["mlp"].sum()
        scale = r["embed"].abs() + r["attn"].abs().sum() + r["mlp"].abs().sum()
        cached_diff = e["answer_logits"][YES_ID].float() - e["answer_logits"][NO_ID].float()
        rows.append({
            "source": label,
            "rel_err_resid": ((total - r["resid_dot"]).abs() / scale).item(),
            "err_logit": (total / r["rms"] - r["logit_diff"]).abs().item(),
            "err_cache": (r["logit_diff"] - cached_diff).abs().item(),
            "answer_match": r["answer_text"].lower().startswith(e["answer_text"].lower()),
            "finite": bool(torch.isfinite(total) and torch.isfinite(r["logit_diff"])),
        })
        results[label].append(r)

print(len(rows), "questions run")

48 questions run


In [10]:
n = len(rows)
for label in samples:
    sub = [r for r in rows if r["source"] == label]
    print(f"{label:18s} rel err resid {max(r['rel_err_resid'] for r in sub):.5f} | "
          f"err logit {max(r['err_logit'] for r in sub):.4f} | err cache {max(r['err_cache'] for r in sub):.4f} | "
          f"answers match {sum(r['answer_match'] for r in sub)}/{len(sub)}")

assert all(r["finite"] for r in rows), "non-finite values"
assert max(r["rel_err_resid"] for r in rows) < 0.05, "components do not sum to the final residual"
assert max(r["err_logit"] for r in rows) < 0.5, "components do not reproduce the logit difference"
assert max(r["err_cache"] for r in rows) < 0.25, "prompt or token ids differ from the question-aware cache"
assert sum(r["answer_match"] for r in rows) == n, "answers differ from the question-aware cache"
print("validation passed")

attribute/amber    rel err resid 0.00012 | err logit 0.0144 | err cache 0.0000 | answers match 12/12
object/pope        rel err resid 0.00012 | err logit 0.0119 | err cache 0.0000 | answers match 12/12
relation/amber     rel err resid 0.00008 | err logit 0.0145 | err cache 0.0000 | answers match 12/12
relation/reefknot  rel err resid 0.00004 | err logit 0.0081 | err cache 0.0000 | answers match 12/12
validation passed


In [11]:
for label, rs in results.items():
    attn = torch.stack([r["attn"] / r["rms"] for r in rs]).mean(0).flatten()
    mlp = torch.stack([r["mlp"] / r["rms"] for r in rs]).mean(0)
    mean_diff = torch.stack([r["logit_diff"] for r in rs]).mean().item()
    print(f"--- {label}: mean logit diff {mean_diff:.3f} ---")
    for i in attn.abs().topk(5).indices.tolist():
        print(f"  layer {i // n_head:2d} head {i % n_head:2d}  {attn[i]:+.3f}")
    for i in mlp.abs().topk(3).indices.tolist():
        print(f"  layer {i:2d} mlp      {mlp[i]:+.3f}")

--- attribute/amber: mean logit diff 1.461 ---
  layer 16 head  0  +0.741
  layer 27 head  7  +0.625
  layer 21 head 14  +0.284
  layer 28 head  7  -0.201
  layer 25 head 28  +0.170
  layer 28 mlp      -2.687
  layer 18 mlp      -2.380
  layer 20 mlp      +2.248
--- object/pope: mean logit diff 1.850 ---
  layer 16 head  0  +0.898
  layer 27 head  7  +0.553
  layer 21 head 14  +0.259
  layer 25 head 28  +0.212
  layer 28 head  7  -0.206
  layer 28 mlp      -2.953
  layer 20 mlp      +2.728
  layer 18 mlp      -1.489
--- relation/amber: mean logit diff -0.095 ---
  layer 16 head  0  +0.653
  layer 27 head  7  +0.410
  layer 28 head  7  -0.275
  layer 21 head 14  +0.183
  layer  9 head  0  +0.149
  layer 18 mlp      -2.463
  layer 28 mlp      -2.218
  layer 20 mlp      +1.502
--- relation/reefknot: mean logit diff 0.556 ---
  layer 19 head 10  -0.621
  layer 17 head 16  +0.514
  layer 16 head  0  +0.467
  layer 27 head  7  +0.466
  layer 26 head  3  +0.409
  layer 20 mlp      +3.281
  la